# K2 · Baseline B0 (zero-shot), B1 (5-shot), B2 (RAG) cho Qwen2.5-3B và LLaMA-3.2-3B (T10)
Model gốc fp16, sinh greedy. ~25–40 phút / model với `N_EVAL = None` (toàn bộ test_synth).

In [ ]:
%%capture
!pip install -q "transformers==4.56.2" "trl==0.21.0" "peft==0.17.0" "bitsandbytes==0.47.0" "accelerate>=1.4" "datasets>=3.0" rank-bm25 optuna

In [ ]:
import sys, glob, json, os, time
sys.path.insert(0, glob.glob('/kaggle/input/**/code/src', recursive=True)[0])
from kimochi.kaggle_env import setup, gpu_info
ROOT = setup()
import pandas as pd
from pathlib import Path
RES = ROOT / 'results'
print(gpu_info())

In [ ]:
N_EVAL = None          # None = toàn bộ; đặt 150 để chạy thử nhanh
EVAL_SETS = ['test_synth'] + [s for s in ['test_gold', 'test_adv'] if (ROOT / f'data/splits/{s}.jsonl').exists()]
MODEL_KEYS = ['qwen', 'llama']
EVAL_SETS

In [ ]:
from kimochi import prompts, metrics, llm, retriever
shots = prompts.pick_shots()
R = retriever.HybridRetriever(retriever.load_docs())
docs_by_id = {d['doc_id']: d for d in R.docs}
GEN = RES / 'gen'; GEN.mkdir(exist_ok=True)

def contexts(q, k=3):
    return [docs_by_id[i] for i in R.search(q, 'bm25_filter', k=k)]

In [ ]:
rows = []
for mk in MODEL_KEYS:
    model, tok, name = llm.load(mk, 'fp16')
    for es in EVAL_SETS:
        ev = prompts.load_eval_set(es)
        ev = ev.head(N_EVAL) if N_EVAL else ev
        for mode in ['zero', 'few', 'rag']:
            msgs = [prompts.build_messages(q, mode, shots=shots, docs=contexts(q) if mode == 'rag' else None) for q in ev['query']]
            preds, lat = llm.generate(model, tok, msgs, bs=8)
            out = ev.assign(pred=preds, latency_batch_s=lat, system=f'{mk}_B{["zero","few","rag"].index(mode)}', eval_set=es)
            out.to_json(GEN / f'{mk}_{mode}_{es}.jsonl', orient='records', lines=True, force_ascii=False)
            s = metrics.summarize(metrics.score_generations(out))
            rows.append({'system': out.system.iloc[0], 'model': name, 'eval_set': es, **s}); print(rows[-1])
    # độ trễ phục vụ thật (batch = 1) trên 30 câu, cấu hình RAG
    q30 = prompts.load_eval_set('test_synth').head(30)['query']
    lat1 = llm.latency_single(model, tok, [prompts.build_messages(q, 'rag', docs=contexts(q)) for q in q30])
    rows.append({'system': f'{mk}_B2_latency', 'model': name, 'eval_set': 'latency30',
                 'latency_p50': float(pd.Series(lat1).median()), 'latency_p95': float(pd.Series(lat1).quantile(.95))})
    del model, tok; llm.free()
base = pd.DataFrame(rows); base.to_csv(RES / 'baseline.csv', index=False); base.round(3)

Kiểm tra định tính vài câu trả lời:

In [ ]:
pd.read_json(GEN / 'qwen_rag_test_synth.jsonl', lines=True)[['query', 'reference', 'pred']].head(8)